In [ ]:
"""
This code/notebook can be used to download Mapillary images in a bounding box specified by the user in lat/lon format.
For every image, detailed labels are also retrieved from Mapillary including the raw-reported poses and open-SfM-corrected poses, and many others.
Other functionalities such as re-trying and tiling are provided to navigate the limits around Mapillary API access.
"""

import mapillary.interface as mly
import os, cv2, shutil, h5py, utm, io, sys
import numpy as np
from PIL import Image, ExifTags
from matplotlib import pyplot as plt
from tqdm import tqdm
import requests
import json, time
from collections import defaultdict

In [ ]:
"""
CONFIGURATION. 
Please change accordingly.
"""

MAPILLARY_TOKEN = 'MLY|XXXXXXXXX' # Get your personal token from Mapillary.com by creating a user account
outpath_groundimgs = "../testregiondownloads/mapillary_images"  # The local path for storing the downloaded Mapillary images
outpath_labels_json = "../testregiondownloads/labels" #  The local path for storing the labels corresponding to the downloaded Mapillary images
mly.set_access_token(MAPILLARY_TOKEN)

# Large area bbox (lon_min, lat_min, lon_max, lat_max)
BIG_BBOX = [4.3171347, 52.0137663, 4.5227971, 53.0983924] # Defines a bounding box in which you would like to download the Mapillary images

# Tile size in lat/lon degrees 
TILE_SIZE = 0.02   # tiling a large area into small tiles due to Mapillary downloading limit
LIMIT = 100        # max number of images per request


In [ ]:
def tile_bbox(bbox, tile_size):
    min_lon, min_lat, max_lon, max_lat = bbox
    tiles = []

    lon = min_lon
    while lon < max_lon:
        lat = min_lat
        while lat < max_lat:
            tiles.append([
                lon,
                lat,
                min(lon + tile_size, max_lon),
                min(lat + tile_size, max_lat)
            ])
            lat += tile_size
        lon += tile_size

    return tiles

def fetch_images_for_tile(bbox, limit=100, max_retries=5):
    images_data = []
    images_ids = []

    url = "https://graph.mapillary.com/images"
    params = {
        "access_token": MAPILLARY_TOKEN,
        "bbox": ",".join(map(str, bbox)),
        "fields": "id,geometry",
        "limit": limit
    }

    while True:
        attempt = 0
        while attempt < max_retries:
            try:
                r = requests.get(url, params=params, timeout=120)
                r.raise_for_status()
                data = r.json()
                break  # successful request, exit retry loop
            except requests.exceptions.HTTPError as e:
                if r.status_code == 500:
                    attempt += 1
                    print(f"500 Server Error for bbox {bbox}, retry {attempt}/{max_retries}")
                    time.sleep(1)
                else:
                    raise e
            except requests.exceptions.RequestException as e:
                # catch other request errors (timeouts, connection errors)
                attempt += 1
                print(f"Request error: {e}, retry {attempt}/{max_retries}")
                time.sleep(1)
        else:
            # all retries failed
            print(f"Skipping bbox {bbox} after {max_retries} failed attempts")
            return images_ids, images_data

        # Process images from this page
        mydata = data.get("data", [])
        for img in mydata:
            img_id = img["id"]

            # Fetch detailed data
            try:
                detaileddata = json.loads(mly.image_from_key(img_id))
                detaileddata["features"]["geometry"] = img["geometry"]
                images_ids.append(img_id)
                images_data.append(detaileddata)
            except Exception as e:
                print(f"Failed to fetch detailed data for image {img_id}: {e}")

        # Pagination
        paging = data.get("paging", {})
        if "next" not in paging:
            break

        url = paging["next"]
        params = None  # next URL already includes all parameters

    return images_ids, images_data

def download_mapillaryimage(url):
    if not url:
        return
    
    r = requests.get(url, stream=True, timeout=20)
    imageStream = io.BytesIO(r.content)
    imageFile = Image.open(imageStream) 

    return imageFile # PIL image

def undistort_image(img_mapil, camera_matrix_mapil, distortion_mapil, image_dimensions, camera_type):
    """ 
    Since in Open-CVL all images provided are undistorted to ease the training of various CVL methods which may or may not rely on distortion parameters.
    """

    if (camera_type=='perspective'):
        print('using perspective model')

        # === Compute undistortion maps ===
        newcameramtx, roi = cv2.getOptimalNewCameraMatrix(camera_matrix_mapil, distortion_mapil, image_dimensions, 0, image_dimensions)
        mapx, mapy = cv2.initUndistortRectifyMap(newcameramtx, distortion_mapil, None, newcameramtx, image_dimensions, cv2.CV_32FC2)
        img_mapil_undist = cv2.remap(img_mapil, mapx, mapy, interpolation=cv2.INTER_LINEAR)
        
    elif (camera_type=='fisheye'):
        print('using fisheye model')

        # === Compute undistortion maps ===
        K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(camera_matrix_mapil, distortion_mapil, image_dimensions, np.eye(3), balance=0.0)
        map1, map2 = cv2.fisheye.initUndistortRectifyMap(camera_matrix_mapil, distortion_mapil, np.eye(3), K_new, image_dimensions, cv2.CV_16SC2)
        img_mapil_undist = cv2.remap(img_mapil, map1, map2, interpolation=cv2.INTER_LINEAR)

    return img_mapil_undist

def apply_exif_orientation(img, orientation):
    """
    Applies EXIF orientation to a PIL image since some Mapillary images are oriented incorrectly.
    """

    if orientation == 1 or orientation is None:
        return img

    elif orientation == 2:
        return img.transpose(Image.FLIP_LEFT_RIGHT)

    elif orientation == 3:
        return img.rotate(180, expand=True)

    elif orientation == 4:
        return img.transpose(Image.FLIP_TOP_BOTTOM)

    elif orientation == 5:
        return img.transpose(Image.FLIP_LEFT_RIGHT).rotate(270, expand=True)

    elif orientation == 6:
        return img.rotate(270, expand=True)

    elif orientation == 7:
        return img.transpose(Image.FLIP_LEFT_RIGHT).rotate(90, expand=True)

    elif orientation == 8:
        return img.rotate(90, expand=True)

    else:
        return img

In [ ]:
def download_and_store_imageswithlabels(img_id, detaileddata, groundimgs_dir_path, labels_json_path):
    try:
        url = detaileddata["features"]["properties"]["thumb_original_url"]
        seq_name = detaileddata["features"]["properties"]["sequence"]   
        mapil_reported_width = detaileddata["features"]["properties"]["width"]   
        mapil_reported_height = detaileddata["features"]["properties"]["height"]   
        lonlat_mapil_raw = detaileddata["features"]["geometry"]["coordinates"]
        heading_mapil_raw = detaileddata["features"]["properties"]["compass_angle"]
        creator_id = detaileddata["features"]["properties"]["creator_id"]
        altitude_mapil_raw = detaileddata["features"]["properties"]["altitude"]
        camera_params_raw = detaileddata["features"]["properties"]["camera_parameters"] 
        lonlat_mapil_corr = detaileddata["features"]["properties"]["computed_geometry"]["coordinates"] # opensfm corrected lonlat
        heading_mapil_corr = detaileddata["features"]["properties"]["computed_compass_angle"] # opensfm corrected heading
        map_orientation_corr = detaileddata["features"]["properties"]["computed_rotation"]
        altitude_mapil_corr = detaileddata["features"]["properties"]["computed_altitude"]  
        camera_type = detaileddata["features"]["properties"]["camera_type"]
    
        url = json.loads(f'"{url}"')
        if not url:
            return
            
        img_mapil = download_mapillaryimage(url)
        exif = img_mapil._getexif()
        if exif is not None: 
            for tag_id, value in exif.items():
                    tag = ExifTags.TAGS.get(tag_id, tag_id)
                    if tag == 'Orientation':
                        img_mapil = apply_exif_orientation(img_mapil, value)

        img_mapil = np.array(img_mapil)
    
        # Define the distortion coefficients
        distortion_mapil = np.zeros(4, np.float32)
        # Define the camera matrix
        fx, fy = camera_params_raw[0]*max(mapil_reported_width, mapil_reported_height) ,  camera_params_raw[0] * max(mapil_reported_width, mapil_reported_height)
        dist1, dist2 = camera_params_raw[1], camera_params_raw[2]
        distortion_mapil[0] = dist1 
        distortion_mapil[1] = dist2 
        cx = int(mapil_reported_width/2.0)
        cy = int(mapil_reported_height/2.0) 
        camera_matrix_mapil = np.array([[fx, 0, cx],
                                [0, fy, cy],
                                [0, 0, 1]], np.float32)
    
        image_dimensions = tuple([mapil_reported_width, mapil_reported_height])    
        img_mapil_undist = undistort_image(img_mapil, camera_matrix_mapil, distortion_mapil, image_dimensions, camera_type)
    
        # Now storing the images and corresponding labels 
        cv2.imwrite(os.path.join(outpath_groundimgs, f'{img_id}.png'), cv2.cvtColor(img_mapil_undist, cv2.COLOR_RGB2BGR))
    
        relevant_labels = {}
        relevant_labels['latlon_mapilraw'] = [lonlat_mapil_raw[1], lonlat_mapil_raw[0]]
        relevant_labels['latlon_mapilopensfm'] = [lonlat_mapil_corr[1], lonlat_mapil_corr[0]]
        relevant_labels['heading_mapilraw'] = heading_mapil_raw
        relevant_labels['heading_mapilopensfm'] = heading_mapil_corr
        relevant_labels['mapil_intrinsics_3x3'] = camera_matrix_mapil.tolist()
        relevant_labels['camera_type'] = camera_type
        relevant_labels['mapil_image_id'] = img_id
        relevant_labels['mapil_sequence_id'] = seq_name
        relevant_labels['mapil_creator_id'] = creator_id # Useful for proper attribution under Mapillary CC-BY-SA. 
    
        with open(os.path.join(outpath_labels_json, f'{img_id}.json'), 'w') as fp:
            json.dump(relevant_labels, fp)
        print(f'Saved Mapillary data corresponding to Mapillary image ID {img_id}')

    except:
        print('Failed to retrieve and store data for Mapillary image ID:', img_id)